# Hands-on 1 – Serverless RL on Modal (CartPole-v1 with PyTorch)
## Colab notebook covering Lesson 1 → Lesson 5 concepts

You’re going to build a **serverless Reinforcement Learning training job** on **Modal** that learns to solve **`CartPole-v1`** using **PyTorch**.

This notebook is designed to be **hands-on** and ties together the primitives from Lessons 1–5:

- **Lesson 1 (Basics):** run your first Modal function and understand `local` vs `remote` execution.
- **Lesson 2 (Scaling + Input Concurrency):** run multiple evaluations in parallel and/or allow multiple inputs per container.
- **Lesson 3 (Images):** define a custom image with `torch` + `gymnasium`, and run build-time steps.
- **Lesson 4 (GPU + CPU/Memory):** request GPU (optional) and reserve CPU/memory for predictable training.
- **Lesson 5 (Volumes):** persist checkpoints + training logs across runs using a Modal Volume (`commit()` / `reload()`).

> **Goal:** Train a DQN agent that reaches a high average return on `CartPole-v1` and persist the best model to a volume.


In [ ]:
# =====================================
# Step 0 – Install and check Modal
# =====================================
!pip install modal --quiet
!which modal
!modal --version
print("✅ Modal installed.")

## Step 1 – Authenticate (required once per machine/runtime)


In [ ]:
# ============================
# Step 1B – Configure Modal using the CLI (matches docs)
# ============================
# ⚠️ IMPORTANT:
# - Replace the placeholder strings with your real MODAL_TOKEN_ID and MODAL_TOKEN_SECRET.
# - Do NOT commit these values to GitHub or share them.
#
# This cell:
#   1. Stores your token via `modal token set`.
#   2. This writes the Modal config file (e.g. ~/.modal.toml) for you.

#TOKEN_ID = "YOUR_TOKEN_ID_HERE"        # <-- paste from Modal dashboard
#TOKEN_SECRET = "YOUR_TOKEN_SECRET_HERE"  # <-- paste from Modal dashboard


TOKEN_ID = ""        # <-- paste from Modal dashboard
TOKEN_SECRET = ""  # <-- paste from Modal dashboard



if "YOUR_TOKEN_ID_HERE" in TOKEN_ID or "YOUR_TOKEN_SECRET_HERE" in TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

# Call the Modal CLI to store the token
!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

print("✅ Token stored via `modal token set`. You should be authenticated now.")

## Step 2 – What we’ll run on Modal

We will generate a single script: **`hands_on_1_cartpole_modal.py`**.

That script defines:
- A **custom image** with the RL dependencies
- A **volume** for saving:
  - `checkpoints/best.pt`
  - `logs/train_metrics.jsonl`
- Modal functions:
  1) `train_dqn(...)` – trains a DQN policy and periodically evaluates
  2) `evaluate_checkpoint(...)` – loads a checkpoint from the volume and evaluates it
  3) `parallel_eval(...)` – runs many evaluations in parallel (**Lesson 2 scaling**)

Then we run it with:
```bash
modal run hands_on_1_cartpole_modal.py --train-episodes 600
```

> **Note:** CartPole can be “solved” in many ways. Here we use a compact DQN implementation that is reliable and easy to read.


In [ ]:
%%writefile hands_on_1_cartpole_modal.py
"""Hands-on 1: CartPole-v1 solved on Modal with PyTorch (DQN).

This file intentionally demonstrates Modal primitives from Lessons 1–5:
- App + functions (Lesson 1)
- Scaling + concurrency (Lesson 2)
- Custom images (Lesson 3)
- GPU / CPU+memory resources (Lesson 4)
- Volumes for persistent storage (Lesson 5)
"""

import json
import os
import random
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple

import modal

# ============================================================
# Modal App (Lesson 1)
# ============================================================
app = modal.App("hands-on-1-cartpole-dqn")


# ============================================================
# Lesson 3 – Custom image (container environment)
# ============================================================
image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "torch==2.4.1",
        "gymnasium==0.29.1",
        "numpy==1.26.4",
    )
    .run_commands(
        "python -c \"import torch, gymnasium as gym; print('torch', torch.__version__); print('gymnasium', gym.__version__)\""
    )
)


# ============================================================
# Lesson 5 – Volume for persistence
# ============================================================
volume = modal.Volume.from_name("hands-on-1-cartpole-volume", create_if_missing=True)
VOL_MOUNT = "/vol"
VOL_DIR = Path(VOL_MOUNT)
CKPT_DIR = VOL_DIR / "checkpoints"
LOG_DIR = VOL_DIR / "logs"
BEST_CKPT_PATH = CKPT_DIR / "best.pt"
METRICS_PATH = LOG_DIR / "train_metrics.jsonl"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except Exception:
        pass


# ============================================================
# DQN implementation (PyTorch)
# ============================================================
@dataclass
class Transition:
    s: List[float]
    a: int
    r: float
    ns: List[float]
    done: bool


class ReplayBuffer:
    def __init__(self, capacity: int = 50_000):
        self.capacity = capacity
        self.data: List[Transition] = []
        self.idx = 0

    def push(self, t: Transition) -> None:
        if len(self.data) < self.capacity:
            self.data.append(t)
        else:
            self.data[self.idx] = t
        self.idx = (self.idx + 1) % self.capacity

    def sample(self, batch_size: int):
        batch = random.sample(self.data, batch_size)
        return batch

    def __len__(self):
        return len(self.data)


def build_qnet(obs_dim: int, n_actions: int):
    import torch.nn as nn
    return nn.Sequential(
        nn.Linear(obs_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
        nn.ReLU(),
        nn.Linear(128, n_actions),
    )


def to_tensor(x, device):
    import torch
    return torch.tensor(x, dtype=torch.float32, device=device)


def evaluate_policy(qnet_state: Dict, episodes: int = 20, seed: int = 0) -> Dict:
    """Evaluate greedy policy for a few episodes on CPU."""
    import gymnasium as gym
    import numpy as np
    import torch

    set_seed(seed)

    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    device = "cpu"
    q = build_qnet(obs_dim, n_actions).to(device)
    q.load_state_dict(qnet_state)
    q.eval()

    returns = []
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        done = False
        total = 0.0
        while not done:
            with torch.no_grad():
                qs = q(to_tensor(s, device).unsqueeze(0))
                a = int(torch.argmax(qs, dim=1).item())
            ns, r, terminated, truncated, _ = env.step(a)
            done = bool(terminated or truncated)
            total += float(r)
            s = ns
        returns.append(total)

    env.close()
    return {
        "episodes": episodes,
        "avg_return": float(np.mean(returns)),
        "min_return": float(np.min(returns)),
        "max_return": float(np.max(returns)),
    }


# ============================================================
# Lesson 4 – Resources (CPU/memory + optional GPU)
# ============================================================
TRAIN_CPU = 4
TRAIN_MEM_MIB = 2048


# ============================================================
# Training function (uses volume + reserved resources)
# ============================================================
@app.function(image=image, cpu=TRAIN_CPU, memory=TRAIN_MEM_MIB, volumes={VOL_MOUNT: volume})
def train_dqn(
    train_episodes: int = 600,
    seed: int = 0,
    eval_every: int = 50,
    eval_episodes: int = 20,
    target_avg_return: float = 475.0,
) -> Dict:
    """Train DQN and persist best checkpoint + logs to Modal Volume."""
    import gymnasium as gym
    import torch
    import torch.nn.functional as F
    import torch.optim as optim

    set_seed(seed)

    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    device = "cpu"  # CartPole is fast on CPU

    q = build_qnet(obs_dim, n_actions).to(device)
    target = build_qnet(obs_dim, n_actions).to(device)
    target.load_state_dict(q.state_dict())
    target.eval()

    optimizer = optim.Adam(q.parameters(), lr=1e-3)
    rb = ReplayBuffer(capacity=50_000)

    gamma = 0.99
    batch_size = 128
    start_learning = 1_000
    update_target_every = 500  # steps
    eps_start, eps_end, eps_decay = 1.0, 0.05, 20_000  # decay in steps

    # Ensure volume directories exist
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    best_avg = -1e9
    global_steps = 0
    t0 = time.time()

    def epsilon(step: int) -> float:
        import math
        return eps_end + (eps_start - eps_end) * math.exp(-step / eps_decay)

    def log_metrics(row: Dict) -> None:
        with METRICS_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")

    for ep in range(1, train_episodes + 1):
        s, _ = env.reset(seed=seed + ep)
        done = False
        ep_return = 0.0

        while not done:
            eps = epsilon(global_steps)
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    qs = q(to_tensor(s, device).unsqueeze(0))
                    a = int(torch.argmax(qs, dim=1).item())

            ns, r, terminated, truncated, _ = env.step(a)
            done = bool(terminated or truncated)
            ep_return += float(r)

            rb.push(Transition(s=list(map(float, s)), a=int(a), r=float(r), ns=list(map(float, ns)), done=done))
            s = ns
            global_steps += 1

            # Learn from replay
            if len(rb) >= start_learning:
                batch = rb.sample(batch_size)
                bs = to_tensor([t.s for t in batch], device)
                ba = torch.tensor([t.a for t in batch], dtype=torch.int64, device=device).unsqueeze(1)
                br = torch.tensor([t.r for t in batch], dtype=torch.float32, device=device).unsqueeze(1)
                bns = to_tensor([t.ns for t in batch], device)
                bdone = torch.tensor([t.done for t in batch], dtype=torch.float32, device=device).unsqueeze(1)

                qsa = q(bs).gather(1, ba)
                with torch.no_grad():
                    max_next = target(bns).max(dim=1, keepdim=True).values
                    y = br + gamma * (1.0 - bdone) * max_next

                loss = F.smooth_l1_loss(qsa, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(q.parameters(), 10.0)
                optimizer.step()

                if global_steps % update_target_every == 0:
                    target.load_state_dict(q.state_dict())

        # Evaluate periodically + checkpoint
        if (ep % eval_every == 0) or (ep == train_episodes):
            q_state = {k: v.detach().cpu() for k, v in q.state_dict().items()}
            eval_res = evaluate_policy(q_state, episodes=eval_episodes, seed=seed + 10_000 + ep)

            row = {
                "ts_utc": utc_now(),
                "episode": ep,
                "train_episode_return": ep_return,
                "eval_avg_return": eval_res["avg_return"],
                "global_steps": global_steps,
                "elapsed_sec": round(time.time() - t0, 2),
            }
            log_metrics(row)

            if eval_res["avg_return"] > best_avg:
                best_avg = eval_res["avg_return"]
                torch.save(q_state, BEST_CKPT_PATH)
                # Lesson 5: commit after writes
                volume.commit()

            if best_avg >= target_avg_return:
                break

    env.close()
    return {
        "status": "ok",
        "best_eval_avg_return": round(float(best_avg), 2),
        "episodes_ran": ep,
        "checkpoint_path": str(BEST_CKPT_PATH),
        "metrics_path": str(METRICS_PATH),
        "cpu_reserved": TRAIN_CPU,
        "memory_mib_reserved": TRAIN_MEM_MIB,
    }


# ============================================================
# Read/evaluate functions (volume commit/reload demo)
# ============================================================
@app.function(image=image, volumes={VOL_MOUNT: volume})
def evaluate_checkpoint(reload: bool = False, episodes: int = 50, seed: int = 123) -> Dict:
    if reload:
        volume.reload()

    if not BEST_CKPT_PATH.exists():
        return {"status": "error", "message": "No checkpoint found yet. Run train_dqn first."}

    import torch
    q_state = torch.load(BEST_CKPT_PATH, map_location="cpu")
    res = evaluate_policy(q_state, episodes=episodes, seed=seed)
    res.update({"status": "ok", "checkpoint": str(BEST_CKPT_PATH)})
    return res


@app.function(image=image, volumes={VOL_MOUNT: volume})
def read_metrics_tail(reload: bool = False, n: int = 10) -> List[Dict]:
    if reload:
        volume.reload()

    if not METRICS_PATH.exists():
        return []

    rows = []
    with METRICS_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows[-n:]


# ============================================================
# Lesson 2 – Scaling + input concurrency demos
# ============================================================
@app.function(image=image, volumes={VOL_MOUNT: volume})
def parallel_eval(num_jobs: int = 8, episodes_per_job: int = 10, reload: bool = True) -> Dict:
    if reload:
        volume.reload()

    if not BEST_CKPT_PATH.exists():
        return {"status": "error", "message": "No checkpoint found yet."}

    import torch
    q_state = torch.load(BEST_CKPT_PATH, map_location="cpu")

    seeds = [1000 + i * 7 for i in range(num_jobs)]
    results = list(_eval_job.map([(q_state, episodes_per_job, s) for s in seeds]))

    avg_of_avgs = sum(r["avg_return"] for r in results) / len(results)
    return {"status": "ok", "mode": "map", "num_jobs": num_jobs, "episodes_per_job": episodes_per_job, "avg_of_avgs": avg_of_avgs, "jobs": results}


@app.function(image=image)
def _eval_job(args: Tuple[Dict, int, int]) -> Dict:
    q_state, episodes, seed = args
    out = evaluate_policy(q_state, episodes=episodes, seed=seed)
    out["seed"] = seed
    return out


@app.function(image=image)
@modal.concurrent(max_inputs=32, target_inputs=24)
def eval_concurrent(q_state: Dict, episodes: int, seed: int) -> Dict:
    out = evaluate_policy(q_state, episodes=episodes, seed=seed)
    out["seed"] = seed
    return out


@app.function(image=image, volumes={VOL_MOUNT: volume})
def parallel_eval_concurrent(num_jobs: int = 32, episodes_per_job: int = 5, reload: bool = True) -> Dict:
    if reload:
        volume.reload()

    if not BEST_CKPT_PATH.exists():
        return {"status": "error", "message": "No checkpoint found yet."}

    import torch
    q_state = torch.load(BEST_CKPT_PATH, map_location="cpu")

    seeds = [2000 + i * 3 for i in range(num_jobs)]
    results = [eval_concurrent.remote(q_state, episodes_per_job, s) for s in seeds]

    avg_of_avgs = sum(r["avg_return"] for r in results) / len(results)
    return {"status": "ok", "mode": "concurrent", "num_jobs": num_jobs, "episodes_per_job": episodes_per_job, "avg_of_avgs": avg_of_avgs}


# ============================================================
# Entrypoint (Lesson 1 style)
# ============================================================
@app.local_entrypoint()
def hands_on_1_main(
    train_episodes: int = 600,
    seed: int = 0,
    eval_every: int = 50,
    eval_episodes: int = 20,
    do_parallel_eval: bool = True,
    do_concurrent_eval: bool = False,
):
    print("\n=================================================")
    print("Hands-on 1 – CartPole-v1 (PyTorch DQN) on Modal")
    print("=================================================\n")


    train_res = train_dqn.remote(
        train_episodes=train_episodes,
        seed=seed,
        eval_every=eval_every,
        eval_episodes=eval_episodes,
    )
    print("TRAIN RESULT:\n", train_res)

    print("\nMETRICS TAIL (no reload):\n", read_metrics_tail.remote(reload=False, n=5))
    print("\nMETRICS TAIL (with reload):\n", read_metrics_tail.remote(reload=True, n=5))

    print("\nEVAL CHECKPOINT (reload=True):\n", evaluate_checkpoint.remote(reload=True, episodes=50, seed=123))

    if do_parallel_eval:
        print("\nPARALLEL EVAL via map() (fan-out across containers):\n", parallel_eval.remote(num_jobs=8, episodes_per_job=10, reload=True))

    if do_concurrent_eval:
        print("\nPARALLEL EVAL via @modal.concurrent (fewer containers):\n", parallel_eval_concurrent.remote(num_jobs=32, episodes_per_job=5, reload=True))

    print("\n✅ Done. Check Modal dashboard: App logs + Storage → Volumes → hands-on-1-cartpole-volume")


## Step 3 – Run the project on Modal

Start with a smaller run to verify everything works, then increase episodes.

Recommended run:
```bash
modal run hands_on_1_cartpole_modal.py --train-episodes 600
```


In [ ]:
!modal run hands_on_1_cartpole_modal.py --train-episodes 600

In [ ]:
!modal run hands_on_1_cartpole_modal.py --train-episodes 600 --eval-every 50 --eval-episodes 20 --do-parallel-eval


## Step 4 – Inspect persistent artifacts (Volume)

Volume name:
- `hands-on-1-cartpole-volume`

Expected files:
- `checkpoints/best.pt`
- `logs/train_metrics.jsonl`

CLI helpers:
```bash
modal volume list
modal volume ls hands-on-1-cartpole-volume
```


In [ ]:
!modal volume list
!modal volume ls hands-on-1-cartpole-volume || true


## Step 5 – Debug inside a container

```bash
modal shell hands_on_1_cartpole_modal.py::train_dqn
```

Inside:
- `ls -la /vol`
- `tail -n 5 /vol/logs/train_metrics.jsonl`


In [ ]:
!modal shell hands_on_1_cartpole_modal.py::train_dqn
